In [ ]:
import cv2
import numpy as np
import pytesseract
import easyocr
from PIL import Image
from datasets import load_dataset

In [ ]:
def pick_channel(image: np.ndarray, channel_name: str = 'all') -> np.ndarray:
    """
    Extracts a specific color channel from an image array.
    You can add more configurations if you need.
    """
    if channel_name == 'all':
        return image
    b, g, r = cv2.split(image)
    if channel_name == 'r':
        return r
    elif channel_name == 'g':
        return g
    elif channel_name == 'b':
        return b
    elif channel_name == 'bg':
        zeros = np.zeros(image.shape[:2], dtype="uint8")
        return cv2.merge([b, g, zeros])
    else:
        raise ValueError(f"Invalid channel name: '{channel_name}'.\
        Choose from 'all', 'r', 'g', 'b', 'bg'.")

In [ ]:
def enhance_image(image: np.ndarray) -> np.ndarray:
    """
    Applies basic enhancements to an image for OCR.
    You can add whatever enhancements you choose to,
    create logic to selectively apply enhancements.
    """
    if len(image.shape) == 3:
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray_image = image
    filtered_image = cv2.bilateralFilter(gray_image, 9, 75, 75)
    thresh_image = cv2.adaptiveThreshold(filtered_image, 255,
                                         cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                         cv2.THRESH_BINARY, 11, 2)
    return thresh_image

In [ ]:
def ocr_with_tesseract(image: np.ndarray) -> str:
    """
    Performs OCR on an image using pytesseract with the Chinese language model.
    """
    print("\n--- Running Tesseract OCR (Chinese) ---")
    pil_image = Image.fromarray(image)
    try:
        # --- MODIFICATION ---
        # Added lang='chi_sim' to specify the Simplified Chinese language model.
        text = pytesseract.image_to_string(pil_image, lang='chi_sim')
        return text
    except Exception as e:
        return f"Tesseract Error: {e}"

In [ ]:
def ocr_with_easyocr(image: np.ndarray) -> str:
    """
    Performs OCR on an image using EasyOCR with Chinese and English models.
    """
    print("\n--- Running EasyOCR (Chinese + English) ---")
    try:
        # --- MODIFICATION ---
        # Changed ['en'] to ['ch_sim', 'en'] to detect both Simplified Chinese
        # and English text (useful for letters, numbers, etc. on the ID card).
        # On first run, this will download the Chinese model.
        reader = easyocr.Reader(['ch_sim', 'en'])
        results = reader.readtext(image)
        extracted_text = " ".join([item[1] for item in results])
        return extracted_text
    except Exception as e:
        return f"EasyOCR Error: {e}"

In [ ]:
print("--- Loading dataset 'lansinuote/ocr_id_card' ---")
try:
    ds = load_dataset("lansinuote/ocr_id_card", split='train')
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Failed to load dataset: {e}")
    exit()

In [ ]:
# Define how many examples from the dataset you want to process
num_examples_to_process = 3
# Loop through the first few examples
for i in range(num_examples_to_process):
    print(f"\n=================================================")
    print(f"===== Processing Example {i + 1}/{num_examples_to_process} =====")
    print(f"=================================================\n")
    pil_image = ds[i]['image']
    image_np_bgr = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    # Step 1: Pick Channel
    # Add logic to switch between channels
    channel_to_process = 'g'
    print(f"--- 1. Picking Channel: '{channel_to_process}' ---")
    channel_image = pick_channel(image_np_bgr, channel_name=channel_to_process)
    # Step 2: Enhance Image
    # Add logic to switch between enhancements
    print("--- 2. Enhancing the selected channel image ---")
    enhanced_image = enhance_image(channel_image)
    # Step 3: Run OCR Engines
    # This is just a sample, you need not run EasyOCR and Tesseract one after
    # the other for every example.
    print("--- 3. Running OCR on the final image ---")
    # Run OCR with Tesseract
    tesseract_text = ocr_with_tesseract(enhanced_image)
    print("\n[Tesseract Output]")
    print(tesseract_text)
    # Run OCR with EasyOCR
    easyocr_text = ocr_with_easyocr(enhanced_image)
    print("\n[EasyOCR Output]")
    print(easyocr_text)